# Set Up

In [ ]:
import pandas as pd
import numpy as np
from pcj.utils import dados
from pcj.processamento import decompor_estatisticas
from pcj.modelos_ML import isolation_forest
from pcj.outliers import (detect_outliers_zscore, detect_outliers_iqr, detect_outliers_pca_kmeans,
                          detect_outliers_lof, detect_outliers_knn, detect_outliers_ensemble,
                          detect_outliers_estatisticos, comparar_marcacoes, resumo_por_mes)

In [ ]:
df, metadados, variaveis = dados(r"data\processed\dados_limpos.xlsx")
df_med = decompor_estatisticas(df, estatistica="med")


Verificação do efeito da transformação logarítmica

In [ ]:
df_med[variaveis].skew()

In [ ]:
np.log1p(df_med[variaveis]).skew()

# Z-Score

In [ ]:
limites_z, total_z, mascara_z = detect_outliers_zscore(df_med, variaveis, verbose=True)
limites_z


In [ ]:
limites_z_log, total_z_log, mascara_z_log = detect_outliers_zscore(df_med, variaveis, log=True, verbose=True)
limites_z_log


In [ ]:
_, total_zscore_log, mascara_zscore_log = detect_outliers_zscore(df_med, variaveis, log=True)
resumo_por_mes(df_med, mascara_zscore_log)


# IQR

In [ ]:
limites_iqr, total_iqr, mascara_iqr = detect_outliers_iqr(df_med, variaveis, verbose=True)
limites_iqr


In [ ]:
resumo_por_mes(df_med, mascara_iqr)


In [ ]:
limites_iqr_log, total_iqr_log, mascara_iqr_log = detect_outliers_iqr(df_med, variaveis, log=True, verbose=True)
limites_iqr_log


In [ ]:
resumo_por_mes(df_med, mascara_iqr_log)

# K-Means
PCA + K-Means em grade: 6 contaminações × 30 seeds = 180 rodadas. Só avalia os meses completos (sem NaN).

In [ ]:
freq_km, cons_km = detect_outliers_pca_kmeans(df_med, variaveis, verbose=True)
df_med.loc[freq_km > 0, ["Data"]].assign(frequencia=freq_km, consenso=cons_km).sort_values("frequencia", ascending=False)

In [ ]:
print(cons_km.dtype)                       # esperado: bool
print(cons_km.index.equals(df_med.index))  # esperado: True
print(freq_km.notna().sum())               # quantos meses foram avaliados?

# Isolation Forest
Grade: 6 contaminações (0,05 a 0,10) × 30 seeds = 180 rodadas. `frequencia` = fração das rodadas em que o mês foi marcado; `consenso` = frequência ≥ 0,5.

In [ ]:
freq_if, cons_if = isolation_forest(df_med, variaveis, verbose=True)
df_med.loc[freq_if > 0, ["Data"]].assign(frequencia=freq_if, consenso=cons_if).sort_values("frequencia", ascending=False)

# LOF
Grade: as mesmas 6 contaminações, sem seeds (o LOF não tem aleatoriedade). Só avalia os meses completos (sem NaN).

In [ ]:
freq_lof, cons_lof = detect_outliers_lof(df_med, variaveis, verbose=True)
df_med.loc[freq_lof > 0, ["Data"]].assign(frequencia=freq_lof, consenso=cons_lof).sort_values("frequencia", ascending=False)

# KNN
Score = distância ao 5º vizinho mais próximo. Mesma grade e mesmos meses avaliados do LOF.

In [ ]:
freq_knn, cons_knn = detect_outliers_knn(df_med, variaveis, verbose=True)
df_med.loc[freq_knn > 0, ["Data"]].assign(frequencia=freq_knn, consenso=cons_knn).sort_values("frequencia", ascending=False)

## LOF × KNN
Os dois marcam os mesmos meses? (linhas: LOF; colunas: KNN)

In [ ]:
pd.crosstab(cons_lof, cons_knn, rownames=["LOF"], colnames=["KNN"])

# Ensemble (IF + LOF + KNN)
Padrão: voto por maioria entre os detectores que avaliaram o mês. Nos meses com NaN, só o IF avalia.

In [ ]:
tabela_ens = detect_outliers_ensemble(df_med, variaveis, verbose=True)
tabela_ens[tabela_ens["votos"] > 0]

Cada detector × ensemble. Os detectores votaram no ensemble: a tabela mostra a contribuição de cada um, não uma comparação independente.

In [ ]:
comparar_marcacoes(tabela_ens, referencia="ensemble")

## Cenários do ensemble
Meses marcados em cada combinação de detectores × regra. Com o K-Means (4 detectores), a maioria exige 3 votos: o empate de 2 a 2 não marca. Cada cenário roda os detectores de novo (a célula leva alguns minutos).

In [ ]:
# Chave: (detectores, regra). Valor: marcação do ensemble (True/False para cada mês)
cenarios = {}

for metodos in [("IF", "LOF", "KNN"), ("IF", "LOF"), ("IF", "LOF", "KNN", "K-Means")]:
    for regra in ["maioria", "unanimidade", "qualquer", "media"]:
        tabela = detect_outliers_ensemble(df_med, variaveis, metodos=metodos, regra=regra)
        cenarios[(" + ".join(metodos), regra)] = tabela["ensemble"]

# Tabela: uma linha por mês, uma coluna por cenário
cenarios = pd.DataFrame(cenarios)

# Meses marcados em cada cenário (linhas: detectores; colunas: regra)
cenarios.sum().unstack()[["maioria", "unanimidade", "qualquer", "media"]]

Em quantos dos 12 cenários cada mês foi marcado (12 = marcado em todos).

In [ ]:
n_cenarios = cenarios.sum(axis=1)
df_med.loc[n_cenarios > 0, ["Data"]].assign(n_cenarios=n_cenarios).sort_values("n_cenarios", ascending=False)

Cada cenário × ensemble padrão (IF + LOF + KNN, maioria).

In [ ]:
comparar_marcacoes(cenarios, referencia=("IF + LOF + KNN", "maioria"))

# Comparação: ensemble × Z-Score, IQR e K-Means
Ensemble padrão (IF + LOF + KNN, maioria): sem o K-Means, para a comparação com ele não ser circular. Z-Score e IQR em escala log; o mês conta quando pelo menos 1 variável foi marcada.

In [ ]:
# Uma linha por mês: Z-Score e IQR, mais as marcações do ensemble e do K-Means
comparacao = detect_outliers_estatisticos(df_med, variaveis, log=True, min_variaveis=1, verbose=True)
comparacao["ensemble"] = tabela_ens["ensemble"]
comparacao["K-Means"] = cons_km

# Quantos dos quatro métodos marcaram cada mês (True vale 1)
comparacao["n_metodos"] = comparacao[["ensemble", "Z-Score", "IQR", "K-Means"]].sum(axis=1)

comparar_marcacoes(comparacao, referencia="ensemble", metodos=["Z-Score", "IQR", "K-Means"])

In [ ]:
comparacao[comparacao["n_metodos"] > 0].sort_values(["n_metodos", "Data"], ascending=[False, True])

## Meses marcados por ano

In [ ]:
comparacao.groupby(comparacao["Data"].dt.year)[["ensemble", "Z-Score", "IQR", "K-Means"]].sum()

# Tabela-resumo
Meses marcados por método. Percentual sobre todos os meses; `avaliados` = meses que o método consegue avaliar.

In [ ]:
# O Z-Score e o IQR avaliam valor a valor: basta o mês ter pelo menos uma variável sem NaN
meses_com_algum_valor = df_med[variaveis].notna().any(axis=1).sum()

# Uma linha por método: meses marcados e meses avaliados
tabela_resumo = {
    "Z-Score (log, ≥ 1 variável)": {"meses": comparacao["Z-Score"].sum(), "avaliados": meses_com_algum_valor},
    "IQR (log, ≥ 1 variável)": {"meses": comparacao["IQR"].sum(), "avaliados": meses_com_algum_valor},
    "PCA + K-Means": {"meses": cons_km.sum(), "avaliados": freq_km.notna().sum()},
    "Isolation Forest": {"meses": cons_if.sum(), "avaliados": freq_if.notna().sum()},
    "LOF": {"meses": cons_lof.sum(), "avaliados": freq_lof.notna().sum()},
    "KNN": {"meses": cons_knn.sum(), "avaliados": freq_knn.notna().sum()},
    "Ensemble (maioria)": {"meses": tabela_ens["ensemble"].sum(), "avaliados": (tabela_ens["avaliaram"] > 0).sum()},
}
tabela_resumo = pd.DataFrame.from_dict(tabela_resumo, orient="index")
tabela_resumo[f"% de {len(df_med)}"] = (100 * tabela_resumo["meses"] / len(df_med)).round(1)
tabela_resumo

# Apêndice
Sensibilidade da comparação do Z-Score e do IQR com o ensemble: escala (log × original) e regra do mês (≥ 1 × ≥ 2 variáveis). A primeira combinação repete a seção Comparação.

In [ ]:
# Chave: (escala, regra do mês). Valor: comparação com o ensemble
resumos = {}

for log in [True, False]:
    for min_variaveis in [1, 2]:
        tabela = detect_outliers_estatisticos(df_med, variaveis, log=log, min_variaveis=min_variaveis)
        tabela["ensemble"] = tabela_ens["ensemble"]
        escala = "log" if log else "original"
        resumos[(escala, f"≥ {min_variaveis} variável(is)")] = comparar_marcacoes(tabela, metodos=["Z-Score", "IQR"])

pd.concat(resumos, names=["escala", "regra do mês", "método"])